In [ ]:
"""
====================================================================================
Blocknative Dataset Preparation and Merging Pipeline
====================================================================================

Purpose
-------
This module performs end-to-end processing of Ethereum transaction data obtained
from Blocknative's historical mempool API archives. The workflow includes:

1. Download: Retrieve 24 hourly `.csv.gz` slices from Blocknative for a specified date.
2. Extraction: Transform `.csv.gz` files into standard `.csv` format using command-line
   decompression tools (e.g., gzip, gunzip).
3. Merging: Read and concatenate all `.csv` files into a single DataFrame for further analysis.

Context
-------
- Each file represents pending Ethereum transactions captured at the mempool layer.
- The merged dataset is suitable for tasks such as MEV detection, Sybil address analysis,
  and DeFi transaction pattern studies.
- Datasets can be large; efficient memory handling is necessary during merging.

Inputs
------
- Folder: `./11_18_2021_data/`
- Files: `00.csv`, `01.csv`, ..., `23.csv` (originally `.csv.gz` from Blocknative)

Output
------
- A single merged CSV file named `merged_2021_11_18.csv` saved in the same directory.

Bash Script Used to Download from Blocknative
---------------------------------------------
```bash
#!/bin/bash

DATE="20240324"
DOMAIN="https://archive.blocknative.com/"
BASE_URL="${DOMAIN}${DATE}/"
SUCCESSFUL_DOWNLOADS=0

for i in $(seq 0 23); do
    HOUR=$(printf "%02d" $i)
    URL="${BASE_URL}${HOUR}.csv.gz"
    FILENAME="${HOUR}.csv.gz"
    RETRIES=0

    while true; do
        HTTP_STATUS=$(curl -o "$FILENAME" -w "%{http_code}" -s "$URL")

        if [ "$HTTP_STATUS" -eq 200 ]; then
            echo "Downloaded $FILENAME"
            ((SUCCESSFUL_DOWNLOADS++))
            break
        elif [ "$HTTP_STATUS" -eq 429 ] || [ "$HTTP_STATUS" -eq 504 ]; then
            echo "Received $HTTP_STATUS. Retrying..."
            sleep 1
            ((RETRIES++))
            if [ $RETRIES -ge 3 ]; then
                echo "Retry limit reached for $FILENAME"
                break
            fi
        elif [ "$HTTP_STATUS" -eq 404 ]; then
            echo "File not found: $FILENAME"
            break
        else
            echo "Download error $HTTP_STATUS for $FILENAME"
            rm -f "$FILENAME"
            break
        fi
    done
done

"""


In [ ]:
import os
import pandas as pd
from tqdm import tqdm

folder_path = "./11_18_2021_data/"
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
dfs = []

for file in tqdm(csv_files, desc="Merging CSV files"):
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path, sep='\t')
    dfs.append(df)

merged_df = pd.concat(dfs, ignore_index=True)
output_path = os.path.join(folder_path, "merged_2021_11_18.csv")
merged_df.to_csv(output_path, index=False)
